## Downloading Dataset from Kaggle

In [ ]:
# import kagglehub

# path = kagglehub.dataset_download("harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows")

# print("Path to dataset files:", path)

## Imports

In [ ]:
import pandas as pd
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('averaged_perceptron_tagger')

## 1. Data Pre-processing

In [ ]:
# Loading Dataset
df = pd.read_csv(r"./dataset/imdb_top_1000.csv")
df.head(2)

In [ ]:
# Filtering Dataset

# Dropping unwanted columns
unwanted_columns = ["Poster_Link"]
df.drop(columns=unwanted_columns, inplace=True)

# Handling Missing Value
df["Meta_score"].fillna(df["Meta_score"].median(), inplace=True)
df["Gross"] = df["Gross"].astype(str).replace(',', '', regex=False).str.replace('$', '', regex=False).str.strip()
df["Gross"] = pd.to_numeric(df["Gross"], errors='coerce')
df["Gross"].fillna(0, inplace=True)
df["Certificate"].fillna("Unknown", inplace=True)
df["Released_Year"] = pd.to_numeric(df["Released_Year"], errors='coerce')
df["Released_Year"].fillna(df["Released_Year"].median(), inplace=True)

In [ ]:
# Formatting Data
text_columns = ["Series_Title", "Certificate", "Genre", "Overview", "Director", "Star1", "Star2", "Star3", "Star4"]
for col in text_columns:
    df[col] = df[col].str.lower()

df['Runtime'] = df['Runtime'].str.replace(' min', '').astype(int)

In [ ]:
# Tokenization
df["Temp"] = df[text_columns].astype(str).agg(" ".join, axis=1)
df["Tokens"] = df["Temp"].apply(word_tokenize)
df.drop(columns=["Temp"], inplace=True)
df.head()

In [ ]:
# Filtering Tokens
punctuation = set(string.punctuation)
stop_words = set(stopwords.words('english'))

filter_set = stop_words.union(punctuation)

def filter_tokens(tkn):
    filtered_words = [word for word in tkn if word not in filter_set and word.isalpha()]
    return filtered_words

df["Filtered_Tokens"] = df["Tokens"].apply(filter_tokens)
df["Filtered_Tokens"].head()

In [ ]:
# Lemmatization
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
    
def lemmatize_tokens(tkns):
    pos_tags = nltk.pos_tag(tkns)
    lemmatized_words = []
    
    for word, tag in pos_tags:
        wntag = get_wordnet_pos(tag)
        lemma = lemmatizer.lemmatize(word, wntag)
        lemmatized_words.append(lemma)
    
    return lemmatized_words

df["Lemmatized_Tokens"] = df["Filtered_Tokens"].apply(lemmatize_tokens)
df["Lemmatized_Tokens"].head()

## 2. Feature Extraction

In [ ]:
# [Text Feature Extraction] Applying TF-IDF => Converting text -> numerical features
df["Clean_Text"] = df["Lemmatized_Tokens"].apply(lambda x: " ".join(x))

tfidf_vectorizer = TfidfVectorizer(min_df=5)

X_tfidf = tfidf_vectorizer.fit_transform(df["Clean_Text"])
X_tfidf.shape

In [ ]:
# [Numeric Feature Extraction]
numeric_columns = ["Released_Year", "Runtime", "Meta_score", "No_of_Votes", "Gross"]

scaler = MinMaxScaler()
X_num_scaled = scaler.fit_transform(df[numeric_columns])
X_num_scaled_sparse = csr_matrix(X_num_scaled)

# Separating X and y
threshold = df["IMDB_Rating"].median()
X = hstack([X_tfidf, X_num_scaled_sparse])
y = (df["IMDB_Rating"] > threshold).astype(int)

## 3. Data Splitting & Model Building

In [ ]:
# Splitting Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Models
lr = LogisticRegression()
nb = MultinomialNB()
svm = LinearSVC()

# Training
lr.fit(X_train, y_train)
nb.fit(X_train, y_train)
svm.fit(X_train, y_train)

## 4. Model Evaluation

In [ ]:
# Testing
y_pred_lr = lr.predict(X_test)
y_pred_nb = nb.predict(X_test)
y_pred_svm = svm.predict(X_test)

# Calculating Scores
lr_acc = accuracy_score(y_test, y_pred_lr)
lr_pre = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)

nb_acc = accuracy_score(y_test, y_pred_nb)
nb_pre = precision_score(y_test, y_pred_nb)
nb_recall = recall_score(y_test, y_pred_nb)
nb_f1 = f1_score(y_test, y_pred_nb)

svm_acc = accuracy_score(y_test, y_pred_svm)
svm_pre = precision_score(y_test, y_pred_svm)
svm_recall = recall_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm)

# Printing
print("\nLogistic Regression Performance")
print(f"Accuracy: {lr_acc:.2f}")
print(f"Precision: {lr_pre:.2f}")
print(f"Recall: {lr_recall:.2f}")
print(f"F1: {lr_f1:.2f}")

print("\nNaive Bayes Performance")
print(f"Accuracy: {nb_acc:.2f}")
print(f"Precision: {nb_pre:.2f}")
print(f"Recall: {nb_recall:.2f}")
print(f"F1: {nb_f1:.2f}")

print("\nSVM Performance")
print(f"Accuracy: {svm_acc:.2f}")
print(f"Precision: {svm_pre:.2f}")
print(f"Recall: {svm_recall:.2f}")
print(f"F1: {svm_f1:.2f}")